# Unit commitment with `milp`, knapsack, two-stage scenarios and heuristics

Mixed-integer optimisation with `scipy.optimize.milp`: which generators to switch on, which
projects to build under a budget, how much to hedge when the future is a set of scenarios,
and what to do when exact optimisation is too slow.

Every model is built first at the smallest size where you can print every matrix and check
every number, then scaled up to a real day.

**What's in here**
- `milp` anatomy on a two-variable problem: `c`, `LinearConstraint`, `Bounds`, `integrality`, `res.status`
- the same problem as an LP: why the answers differ
- unit commitment for 2 units × 2 hours: the variable table, every constraint row, the solution
- LP relaxation and the integrality gap; why rounding fails
- the real day: 3 units × 24 hours
- knapsack with 4 items: exact vs greedy
- two-stage hedging with 2 scenarios, then 200, and how to know you have enough
- heuristics: merit order by hand, `differential_evolution` on a 1-D toy
- timing as the horizon grows

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import milp, linprog, LinearConstraint, Bounds, differential_evolution

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
np.set_printoptions(precision=3, suppress=True)

## 1. `milp` anatomy

`milp(c, constraints=LinearConstraint(A, lb, ub), integrality=..., bounds=Bounds(lo, hi))`
minimises `c @ x`. Everything is a matrix that you build yourself.

Toy: minimise `3x + 2y` subject to `x + y ≥ 4.5`, `0 ≤ x ≤ 3`, `y ≥ 0`, and **y must be an integer**.

In [2]:
c = np.array([3.0, 2.0])                  # objective coefficients for (x, y)
A = np.array([[1.0, 1.0]])                # one constraint row: 1*x + 1*y
lb = np.array([4.5])                      # row lower bound  -> x + y >= 4.5
ub = np.array([np.inf])                   # row upper bound  -> none
print("c :", c)
print("A :", A)
print("lb:", lb, " ub:", ub)

c : [3. 2.]
A : [[1. 1.]]
lb: [4.5]  ub: [inf]


In [3]:
constraints = LinearConstraint(A, lb, ub)
integrality = np.array([0, 1])            # 0 = continuous, 1 = integer  (one entry per variable)
bounds = Bounds([0, 0], [3, np.inf])      # variable bounds: x in [0, 3], y in [0, inf)

res = milp(c, constraints=constraints, integrality=integrality, bounds=bounds)
print("status :", res.status, "|", res.message)
print("x, y   :", res.x)
print("cost   :", res.fun)

status : 0 | Optimization terminated successfully. (HiGHS Status 7: Optimal)
x, y   : [0.5 4. ]
cost   : 9.5


`y` is cheaper per unit (2 vs 3), so the solver wants all of the 4.5 from `y`, but `y` must be
whole: `y = 4` and the remaining 0.5 from `x`, cost 4·2 + 0.5·3 = 9.5. (`y = 5` alone would cost 10.)

**Pitfall:** `res.status` 0 = optimal, 1 = time limit, 2 = infeasible, 3 = unbounded. An
infeasible problem still returns an object, with `res.x = None`. Check the status every time.

In [4]:
# x + y >= 10 with x <= 3 and y <= 5: impossible (max is 8)
impossible = milp(c, constraints=LinearConstraint(A, [10.0], [np.inf]), integrality=integrality, bounds=Bounds([0, 0], [3, 5]))
print("status:", impossible.status, "|", impossible.message)
print("x     :", impossible.x)

status: 2 | The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)
x     : None


### The same problem as an LP

Drop the integrality and the solver may use fractions.

In [5]:
res_lp = milp(c, constraints=constraints, integrality=np.array([0, 0]), bounds=bounds)
print("LP  : x, y =", res_lp.x, " cost", res_lp.fun)
print("MILP: x, y =", res.x, " cost", res.fun)

LP  : x, y = [0.  4.5]  cost 9.0
MILP: x, y = [0.5 4. ]  cost 9.5


The LP takes `y = 4.5` for cost 9.0. That is a **lower bound** on the integer problem: no
integer solution can beat it. The difference (9.5 vs 9.0) is the integrality gap.

## 2. Unit commitment: 2 units × 2 hours

Two generators. For each hour we decide how much each produces (`p`), whether it is on (`u`,
binary) and whether it just started (`s`).

In [6]:
units = pd.DataFrame({
    "p_max":    [10.0, 8.0],     # GW
    "p_min":    [ 3.0, 2.0],     # GW when on
    "mc":       [40.0, 90.0],    # cost per GWh produced
    "no_load":  [100.0, 50.0],   # cost per hour when on
    "start_up": [200.0, 30.0],   # cost per start
}, index=["A", "B"])
demand = np.array([12.0, 6.0])   # GW in hour 0 and hour 1
units

,p_max,p_min,mc,no_load,start_up
A,10.0,3.0,40.0,100.0,200.0
B,8.0,2.0,90.0,50.0,30.0


### The variable table

Twelve variables in one vector. Write down the position of each one and print the table;
indexing mistakes are the most common MILP bug.

In [7]:
names = []
for kind in ["p", "u", "s"]:
    for g in ["A", "B"]:
        for h in [0, 1]:
            names.append(f"{kind}_{g}{h}")
var = pd.DataFrame({"name": names})
var.index.name = "column"
var.T

column,0,1,2,3,4,5,6,7,8,9,10,11
name,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1


In [8]:
col = {name: i for i, name in enumerate(names)}    # name -> column number
print("column of p_B1:", col["p_B1"], "  column of u_A0:", col["u_A0"])

column of p_B1: 3   column of u_A0: 4


### The cost vector

Producing costs `mc` per GWh, being on costs `no_load` per hour, starting costs `start_up`.

In [9]:
c = np.zeros(12)
for g in ["A", "B"]:
    for h in [0, 1]:
        c[col[f"p_{g}{h}"]] = units.loc[g, "mc"]
        c[col[f"u_{g}{h}"]] = units.loc[g, "no_load"]
        c[col[f"s_{g}{h}"]] = units.loc[g, "start_up"]
pd.Series(c, index=names).to_frame("cost").T

,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1
cost,40.0,40.0,90.0,90.0,100.0,100.0,50.0,50.0,200.0,200.0,30.0,30.0


### The constraint rows

Each row is one inequality `lb ≤ A_row · x ≤ ub`. Build them one family at a time and print
the matrix with labels.

Family 1, demand: `p_A + p_B = demand` in each hour (equality: `lb = ub`).

In [10]:
rows, row_lb, row_ub, row_names = [], [], [], []

for h in [0, 1]:
    a = np.zeros(12)
    a[col[f"p_A{h}"]] = 1
    a[col[f"p_B{h}"]] = 1
    rows.append(a); row_lb.append(demand[h]); row_ub.append(demand[h]); row_names.append(f"demand h{h}")

pd.DataFrame(rows, columns=names, index=row_names).assign(lb=row_lb, ub=row_ub)

,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1,lb,ub
demand h0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,12.0
demand h1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,6.0


Family 2, capacity: `p ≤ p_max · u`, written as `p − p_max · u ≤ 0`. If the unit is off
(`u = 0`) this forces `p ≤ 0`.

In [11]:
for g in ["A", "B"]:
    for h in [0, 1]:
        a = np.zeros(12)
        a[col[f"p_{g}{h}"]] = 1
        a[col[f"u_{g}{h}"]] = -units.loc[g, "p_max"]
        rows.append(a); row_lb.append(-np.inf); row_ub.append(0); row_names.append(f"p<=pmax*u {g}{h}")

pd.DataFrame(rows[2:6], columns=names, index=row_names[2:6]).assign(lb=row_lb[2:6], ub=row_ub[2:6])

,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1,lb,ub
p<=pmax*u A0,1.0,0.0,0.0,0.0,-10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-inf,0
p<=pmax*u A1,0.0,1.0,0.0,0.0,0.0,-10.0,0.0,0.0,0.0,0.0,0.0,0.0,-inf,0
p<=pmax*u B0,0.0,0.0,1.0,0.0,0.0,0.0,-8.0,0.0,0.0,0.0,0.0,0.0,-inf,0
p<=pmax*u B1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-8.0,0.0,0.0,0.0,0.0,-inf,0


Family 3, minimum output when on: `p ≥ p_min · u`, i.e. `p − p_min · u ≥ 0`.

In [12]:
for g in ["A", "B"]:
    for h in [0, 1]:
        a = np.zeros(12)
        a[col[f"p_{g}{h}"]] = 1
        a[col[f"u_{g}{h}"]] = -units.loc[g, "p_min"]
        rows.append(a); row_lb.append(0); row_ub.append(np.inf); row_names.append(f"p>=pmin*u {g}{h}")

pd.DataFrame(rows[6:10], columns=names, index=row_names[6:10]).assign(lb=row_lb[6:10], ub=row_ub[6:10])

,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1,lb,ub
p>=pmin*u A0,1.0,0.0,0.0,0.0,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,inf
p>=pmin*u A1,0.0,1.0,0.0,0.0,0.0,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0,inf
p>=pmin*u B0,0.0,0.0,1.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0,inf
p>=pmin*u B1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-2.0,0.0,0.0,0.0,0.0,0,inf


Family 4, start-ups: `s_h ≥ u_h − u_{h−1}`, i.e. `s_h − u_h + u_{h−1} ≥ 0`. Both units are off
before hour 0, so for `h = 0` it is `s_0 − u_0 ≥ 0`.

In [13]:
for g in ["A", "B"]:
    for h in [0, 1]:
        a = np.zeros(12)
        a[col[f"s_{g}{h}"]] = 1
        a[col[f"u_{g}{h}"]] = -1
        if h == 1:
            a[col[f"u_{g}0"]] = 1
        rows.append(a); row_lb.append(0); row_ub.append(np.inf); row_names.append(f"start {g}{h}")

A = np.array(rows)
print("A shape:", A.shape, "(14 rows x 12 variables)")
pd.DataFrame(rows[10:14], columns=names, index=row_names[10:14]).assign(lb=row_lb[10:14], ub=row_ub[10:14])

A shape: (14, 12) (14 rows x 12 variables)


,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1,lb,ub
start A0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,inf
start A1,0.0,0.0,0.0,0.0,1.0,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,0,inf
start B0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,1.0,0.0,0,inf
start B1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0,0.0,0.0,0.0,1.0,0,inf


### Integrality and bounds

`u` is binary: integer with bounds [0, 1]. `s` can stay continuous in [0, 1]: the start rows
force it to 1 whenever a unit turns on, and its cost pushes it to 0 otherwise. `p ≥ 0` with
no upper bound (the capacity rows handle that).

In [14]:
integrality = np.zeros(12)
hi = np.full(12, np.inf)
for g in ["A", "B"]:
    for h in [0, 1]:
        integrality[col[f"u_{g}{h}"]] = 1
        hi[col[f"u_{g}{h}"]] = 1
        hi[col[f"s_{g}{h}"]] = 1
pd.DataFrame({"integrality": integrality, "upper bound": hi}, index=names).T

,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1
integrality,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
upper bound,inf,inf,inf,inf,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [15]:
res = milp(c, constraints=LinearConstraint(A, row_lb, row_ub), integrality=integrality, bounds=Bounds(0, hi))
print("status:", res.status, "|", res.message)
print("total cost:", res.fun)
pd.Series(res.x, index=names).round(3).to_frame("value").T

status: 0 | Optimization terminated successfully. (HiGHS Status 7: Optimal)
total cost: 1300.0


,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1
value,10.0,6.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0


Read it: hour 0 needs 12 GW, more than A's 10, so both units run (A at 10, B at 2 = its minimum).
Hour 1 needs 6 GW: A alone. B started once (`s_B0 = 1`), A once (`s_A0 = 1`).

Verify the demand rows by hand from the solution:

In [16]:
p_A = res.x[[col["p_A0"], col["p_A1"]]]
p_B = res.x[[col["p_B0"], col["p_B1"]]]
print("p_A + p_B per hour:", p_A + p_B, " demand:", demand)
print("cost check: production", (res.x[:4] * c[:4]).sum(), "+ no-load", (res.x[4:8] * c[4:8]).sum(), "+ starts", (res.x[8:] * c[8:]).sum(), "=", res.fun)

p_A + p_B per hour: [12.  6.]  demand: [12.  6.]
cost check: production 820.0 + no-load 250.0 + starts 230.0 = 1300.0


## 3. LP relaxation and the integrality gap

Same matrices, `integrality` all zero. `u` may now be a fraction.

In [17]:
res_lp = milp(c, constraints=LinearConstraint(A, row_lb, row_ub), integrality=np.zeros(12), bounds=Bounds(0, hi))
print("LP cost  :", round(res_lp.fun, 2), "   MILP cost:", round(res.fun, 2), "   gap:", f"{res.fun / res_lp.fun - 1:.1%}")
pd.DataFrame({"MILP": res.x, "LP": res_lp.x}, index=names).round(3).T

LP cost  : 1200.0    MILP cost: 1300.0    gap: 8.3%


,p_A0,p_A1,p_B0,p_B1,u_A0,u_A1,u_B0,u_B1,s_A0,s_A1,s_B0,s_B1
MILP,10.0,6.0,2.0,0.0,1.0,1.0,1.00,0.0,1.0,0.0,1.00,0.0
LP,10.0,6.0,2.0,0.0,1.0,0.6,0.25,0.0,1.0,0.0,0.25,0.0


The LP runs B at "25 % on" in hour 0 (`u_B0 = 0.25`), paying a quarter of the no-load and
start-up costs. That does not exist physically. Rounding `u_B0` to 0 leaves hour 0 short of
power; rounding to 1 is just the MILP answer, and in bigger problems rounding is usually
infeasible. The LP value is a bound, not a plan.

In [18]:
hi_round = hi.copy(); lo_round = np.zeros(12)
lo_round[col["u_B0"]] = hi_round[col["u_B0"]] = 0.0          # round 0.25 down to 0, then re-solve for p
res_round = milp(c, constraints=LinearConstraint(A, row_lb, row_ub), integrality=np.zeros(12), bounds=Bounds(lo_round, hi_round))
print("rounded-down LP: status", res_round.status, "|", res_round.message)

rounded-down LP: status 2 | The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)


## 4. The real day: 3 units × 24 hours

The same four families of rows, built in loops. The function below repeats exactly the
steps of section 2 for any number of units and hours, plus one new family, minimum up-time:
if a unit starts at hour `h` it must stay on until `h + min_up − 1`, i.e. `u_{h+k} ≥ s_h`.

In [19]:
units3 = pd.DataFrame({
    "p_max":    [20.0, 12.0, 10.0],
    "p_min":    [ 8.0,  4.0,  2.0],
    "mc":       [40.0, 70.0, 120.0],
    "no_load":  [300.0, 150.0, 50.0],
    "start_up": [500.0, 200.0, 50.0],
    "min_up":   [24, 4, 1],
}, index=["nuclear", "ccgt", "peaker"])

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
day = df[df["time"].dt.date == pd.Timestamp("2023-01-17").date()]
demand24 = day["consumption_mwh"].values / 1000            # GW
print("demand GW, hours 0-23:", demand24.round(1))

demand GW, hours 0-23: [31.4 30.  28.8 28.6 29.  29.8 31.2 34.1 36.  37.  36.5 35.8 34.9 34.5
 33.1 33.3 34.6 37.4 38.9 36.8 35.2 33.2 30.6 29.3]


In [20]:
def build_uc(demand, units):
    G, H = len(units), len(demand)
    names = [f"{k}_{g}{h}" for k in ["p", "u", "s"] for g in units.index for h in range(H)]
    col = {n: i for i, n in enumerate(names)}
    n = len(names)
    c = np.zeros(n)
    rows, lb, ub = [], [], []
    for g in units.index:
        for h in range(H):
            c[col[f"p_{g}{h}"]] = units.loc[g, "mc"]
            c[col[f"u_{g}{h}"]] = units.loc[g, "no_load"]
            c[col[f"s_{g}{h}"]] = units.loc[g, "start_up"]
    for h in range(H):                                              # family 1: demand
        a = np.zeros(n)
        for g in units.index:
            a[col[f"p_{g}{h}"]] = 1
        rows.append(a); lb.append(demand[h]); ub.append(demand[h])
    for g in units.index:
        for h in range(H):
            a = np.zeros(n); a[col[f"p_{g}{h}"]] = 1; a[col[f"u_{g}{h}"]] = -units.loc[g, "p_max"]     # family 2
            rows.append(a); lb.append(-np.inf); ub.append(0)
            a = np.zeros(n); a[col[f"p_{g}{h}"]] = 1; a[col[f"u_{g}{h}"]] = -units.loc[g, "p_min"]     # family 3
            rows.append(a); lb.append(0); ub.append(np.inf)
            a = np.zeros(n); a[col[f"s_{g}{h}"]] = 1; a[col[f"u_{g}{h}"]] = -1                          # family 4
            if h > 0:
                a[col[f"u_{g}{h-1}"]] = 1
            rows.append(a); lb.append(0); ub.append(np.inf)
            for k in range(1, int(units.loc[g, "min_up"])):                                             # family 5: min up-time
                if h + k < H:
                    a = np.zeros(n); a[col[f"u_{g}{h+k}"]] = 1; a[col[f"s_{g}{h}"]] = -1
                    rows.append(a); lb.append(0); ub.append(np.inf)
    integrality = np.zeros(n); hi = np.full(n, np.inf)
    for g in units.index:
        for h in range(H):
            integrality[col[f"u_{g}{h}"]] = 1; hi[col[f"u_{g}{h}"]] = 1; hi[col[f"s_{g}{h}"]] = 1
    return c, LinearConstraint(np.array(rows), lb, ub), integrality, Bounds(0, hi), names

c24, cons24, integ24, bnds24, names24 = build_uc(demand24, units3)
print("variables:", len(names24), " constraint rows:", cons24.A.shape[0])

variables: 216  constraint rows: 582


Check the function against section 2 before trusting it: on the 2 × 2 instance it must give
the same cost.

In [21]:
c2, cons2, integ2, bnds2, names2 = build_uc(demand, units.assign(min_up=1))
check = milp(c2, constraints=cons2, integrality=integ2, bounds=bnds2)
print("function on the 2x2 toy:", check.fun, "  hand-built version:", res.fun)

function on the 2x2 toy: 1300.0   hand-built version: 1300.0


In [22]:
t0 = time.time()
res24 = milp(c24, constraints=cons24, integrality=integ24, bounds=bnds24)
print("status:", res24.status, "|", res24.message, f"| {time.time() - t0:.2f} s")
print(f"total cost: {res24.fun:,.0f}")

x = pd.Series(res24.x, index=names24)
on = pd.DataFrame({g: x[[f"u_{g}{h}" for h in range(24)]].values for g in units3.index}).round().astype(int)
prod = pd.DataFrame({g: x[[f"p_{g}{h}" for h in range(24)]].values for g in units3.index})
on.T

status: 0 | Optimization terminated successfully. (HiGHS Status 7: Optimal) | 0.00 s
total cost: 56,604


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
nuclear,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
ccgt,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
peaker,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0


In [23]:
print("starts per unit:", {g: int(round(x[[f"s_{g}{h}" for h in range(24)]].sum())) for g in units3.index})
print("demand met, max gap GW:", np.abs(prod.sum(axis=1).values - demand24).max().round(6))
print("output when off, max GW:", (prod.values * (1 - on.values)).max().round(6))

starts per unit: {'nuclear': 1, 'ccgt': 1, 'peaker': 1}
demand met, max gap GW: 0.0
output when off, max GW: 0.0


The peaker runs one daytime block (one start-up), the CCGT runs all day. Always print these
sanity checks: demand met every hour, zero output when off.

The LP relaxation and gap for the real day:

In [24]:
res24_lp = milp(c24, constraints=cons24, integrality=np.zeros(len(names24)), bounds=bnds24)
print(f"MILP {res24.fun:,.0f}   LP {res24_lp.fun:,.0f}   gap {res24.fun / res24_lp.fun - 1:.2%}")
print("fractional peaker u in the LP, hours 0-11:", pd.Series(res24_lp.x, index=names24)[[f"u_peaker{h}" for h in range(12)]].values.round(2))

MILP 56,604   LP 55,871   gap 1.31%
fractional peaker u in the LP, hours 0-11: [0.13 0.   0.   0.   0.   0.   0.   0.21 0.4  0.5  0.45 0.38]


**Pitfall:** big-M. Writing `p ≤ 1e6 · u` instead of `p ≤ p_max · u` is also correct, but it
makes the LP relaxation useless (`u` can be tiny) and the solver slow. Use the real capacity.

## 5. Knapsack: which projects to build

Four projects, capex and yearly margin, a budget of 100. Choose the set with the largest
margin: all-binary `milp`, minimising the *negative* margin.

In [25]:
proj = pd.DataFrame({"capex": [60, 50, 40, 30], "margin": [11, 9, 8, 5]}, index=["wind", "solar", "battery", "ccgt"])
proj["margin_per_capex"] = (proj["margin"] / proj["capex"]).round(3)
budget = 100
proj

,capex,margin,margin_per_capex
wind,60,11,0.183
solar,50,9,0.180
battery,40,8,0.200
ccgt,30,5,0.167


In [26]:
c_kn = -proj["margin"].values                                   # minimise -margin = maximise margin
A_kn = proj["capex"].values.reshape(1, -1)                      # one row: total capex
print("c :", c_kn)
print("A :", A_kn, " <= ", budget)
kn = milp(c_kn, constraints=LinearConstraint(A_kn, -np.inf, budget), integrality=np.ones(4), bounds=Bounds(0, 1))
proj["chosen"] = kn.x.round().astype(int)
print("status:", kn.status, " total margin:", -kn.fun, " capex used:", (proj["capex"] * proj["chosen"]).sum())
proj

c : [-11  -9  -8  -5]
A : [[60 50 40 30]]  <=  100
status: 0  total margin: 19.0  capex used: 100


,capex,margin,margin_per_capex,chosen
wind,60,11,0.183,1
solar,50,9,0.180,0
battery,40,8,0.200,1
ccgt,30,5,0.167,0


Best: wind + battery, capex exactly 100, margin 19. Greedy by margin-per-capex takes battery
first (0.200), then wind (0.183), and stops: the same answer here. But greedy can fail. Change
the budget to 110:

In [27]:
budget = 110
kn2 = milp(c_kn, constraints=LinearConstraint(A_kn, -np.inf, budget), integrality=np.ones(4), bounds=Bounds(0, 1))
exact = proj.index[kn2.x > 0.5].tolist()

greedy, spent = [], 0
for name in proj.sort_values("margin_per_capex", ascending=False).index:
    if spent + proj.loc[name, "capex"] <= budget:
        greedy.append(name); spent += proj.loc[name, "capex"]
print("exact :", exact, " margin", proj.loc[exact, "margin"].sum())
print("greedy:", greedy, " margin", proj.loc[greedy, "margin"].sum())

exact : ['wind', 'solar']  margin 20
greedy: ['battery', 'wind']  margin 19


Greedy fills up with the best-ratio items and then cannot fit the big one; the exact answer is
better. At this size the MILP is instant, so use it and keep greedy as a sanity check.

## 6. Two-stage hedging with scenarios

Decide **now** a forward volume `F` at price `K`. Later, in each scenario `s` with load `L_s`
and spot `P_s`, the shortfall is bought at `P_s + c_u` and the surplus sold at `P_s − c_o`.
Minimise the expected cost:

`K·F + (1/S) Σ_s [ (P_s + c_u)·short_s − (P_s − c_o)·long_s ]`
subject to `F + short_s − long_s = L_s` for every scenario.

Two scenarios first.

In [28]:
K = 100.0
c_u, c_o = 30.0, 10.0
L_s = np.array([10.0, 14.0])
P_s = np.array([120.0, 80.0])
names2s = ["F", "short_1", "long_1", "short_2", "long_2"]

c_2s = np.array([K, (P_s[0] + c_u) / 2, -(P_s[0] - c_o) / 2, (P_s[1] + c_u) / 2, -(P_s[1] - c_o) / 2])
A_2s = np.array([[1, 1, -1, 0, 0],
                 [1, 0, 0, 1, -1]])
pd.DataFrame(A_2s, columns=names2s, index=["scenario 1", "scenario 2"]).assign(equals=L_s)

,F,short_1,long_1,short_2,long_2,equals
scenario 1,1,1,-1,0,0,10.0
scenario 2,1,0,0,1,-1,14.0


In [29]:
pd.Series(c_2s, index=names2s).to_frame("cost per unit").T

,F,short_1,long_1,short_2,long_2
cost per unit,100.0,75.0,-55.0,55.0,-35.0


Short in scenario 1 costs (120 + 30)/2 = 75 per unit (the /2 is the scenario probability);
long in scenario 2 *earns* (80 − 10)/2 = 35, hence the negative cost.

In [30]:
r2s = linprog(c_2s, A_eq=A_2s, b_eq=L_s, bounds=[(0, 20)] + [(0, None)] * 4, method="highs")
print("status:", r2s.status, "|", r2s.message)
pd.Series(r2s.x, index=names2s).round(3).to_frame("value").T

status: 0 | Optimization terminated successfully. (HiGHS Status 7: Optimal)


,F,short_1,long_1,short_2,long_2
value,14.0,0.0,4.0,0.0,0.0


`F = 14`: hedge the *larger* load, then be 4 long in scenario 1. Being short in scenario 1
would cost 150 per unit against a forward at 100; being long in scenario 2 loses only
100 − 70 = 30 per unit. With asymmetric costs the LP over-hedges, as the newsvendor rule predicts.

**Pitfall:** the LP can be unbounded. If the scenarios' average sell price `P_s − c_o` is above
the forward price `K`, the model thinks buying an infinite forward volume and selling it all
back is free money. Remove the upper bound on `F` with spot scenarios of 120 and 130:

In [31]:
P_hi = np.array([120.0, 130.0])
c_hi = np.array([K, (P_hi[0] + c_u) / 2, -(P_hi[0] - c_o) / 2, (P_hi[1] + c_u) / 2, -(P_hi[1] - c_o) / 2])
print("earn per unit long, scenario 1 and 2:", -c_hi[2] * 2, -c_hi[4] * 2, " vs forward price", K)
r_hi = linprog(c_hi, A_eq=A_2s, b_eq=L_s, bounds=[(0, None)] + [(0, None)] * 4, method="highs")
print("no limit on F : status", r_hi.status, "|", r_hi.message)
r_lim = linprog(c_hi, A_eq=A_2s, b_eq=L_s, bounds=[(0, 20)] + [(0, None)] * 4, method="highs")
print("F limited to 20: status", r_lim.status, " F =", r_lim.x[0])

earn per unit long, scenario 1 and 2: 110.0 120.0  vs forward price 100.0
no limit on F : status 3 | The problem is unbounded. (HiGHS Status 10: model_status is Unbounded; primal_status is At upper bound)
F limited to 20: status 0  F = 20.0


With a position limit the solver simply buys the maximum (20) and the answer sits on the bound,
which is your signal that the scenario set has an arbitrage in it. Real desks have position
limits; always give `F` one, and print whether the solution sits on it.

A second thing to look at: a scenario with a negative price makes being *short* profitable in
that scenario (cost `P_s + c_u` below zero). That is only real if the imbalance price can really
go negative; otherwise clip scenario prices at zero, deliberately.

### Many scenarios from the data

Bootstrap January 2023 hours for a 5 % retailer, solve, and compare `F*` with the load
quantile at `c_u / (c_u + c_o)`.

In [32]:
jan = df[(df["time"] >= "2023-01-01") & (df["time"] < "2023-02-01")]
L_hist = jan["consumption_mwh"].values * 0.05
P_hist = jan["price_eur_mwh"].values
K = P_hist.mean()
c_u, c_o = 25.0, 12.0
F_max = 1.5 * L_hist.max()

def solve_hedge(L, P):
    S = len(L)
    c = np.concatenate([[K], (P + c_u) / S, -(P - c_o) / S])
    A = np.zeros((S, 1 + 2 * S))
    A[:, 0] = 1
    A[np.arange(S), 1 + np.arange(S)] = 1
    A[np.arange(S), 1 + S + np.arange(S)] = -1
    r = linprog(c, A_eq=A, b_eq=L, bounds=[(0, F_max)] + [(0, None)] * (2 * S), method="highs")
    assert r.status == 0, r.message
    return r.x[0], r.fun

def make_scenarios(S, seed):
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(L_hist), S)
    L = L_hist[idx] * r.lognormal(0, 0.05, S)
    P = np.clip(P_hist[idx] * r.lognormal(0, 0.15, S), 0, None)     # no negative prices, deliberately
    return L, P

L200, P200 = make_scenarios(200, 1)
F_star, cost_star = solve_hedge(L200, P200)
print(f"F* = {F_star:,.0f} MWh/h   expected hourly cost {cost_star:,.0f}")
print(f"load mean {L_hist.mean():,.0f}   load quantile at c_u/(c_u+c_o) = {np.quantile(L_hist, c_u / (c_u + c_o)):,.0f}")

F* = 1,728 MWh/h   expected hourly cost 196,492
load mean 1,617   load quantile at c_u/(c_u+c_o) = 1,718


### How many scenarios are enough?

Re-solve with fresh draws and watch how much `F*` moves between draws, and evaluate each
`F*` on a large **independent** scenario set (never on the scenarios it was optimised on).

In [33]:
L_oos, P_oos = make_scenarios(20_000, 999)

def oos_cost(F):
    short = np.maximum(L_oos - F, 0)
    long = np.maximum(F - L_oos, 0)
    return np.mean(K * F + (P_oos + c_u) * short - (P_oos - c_o) * long)

rows = []
for S in [10, 50, 200, 1000]:
    Fs = []
    for seed in range(5):
        L, P = make_scenarios(S, seed)
        Fs.append(solve_hedge(L, P)[0])
    Fs = np.array(Fs)
    rows.append([S, Fs.round(0), Fs.std().round(1), oos_cost(Fs.mean()).round(0)])
pd.DataFrame(rows, columns=["scenarios", "F* for 5 draws", "std of F*", "OOS cost at mean F*"])

,scenarios,F* for 5 draws,std of F*,OOS cost at mean F*
0,10,"[1925.0, 1871.0, 1744.0, 1969.0, 1781.0]",84.8,197815.0
1,50,"[1781.0, 1761.0, 1816.0, 1811.0, 3036.0]",498.1,199394.0
2,200,"[1746.0, 1728.0, 1777.0, 1729.0, 1754.0]",17.9,197423.0
3,1000,"[1734.0, 1737.0, 1748.0, 1742.0, 1728.0]",6.8,197423.0


With 10 scenarios `F*` jumps around by hundreds of MWh between draws; with 1,000 it is stable.
One of the 50-scenario draws hit the position limit (3,036 = `F_max`): in that draw the average
sell price happened to exceed `K`, the free-money case from the pitfall above. Convergence is not
monotonic with small samples. The out-of-sample cost is flat once `S` is a few hundred: the cost
is not very sensitive to `F` near the optimum, so there is no point over-engineering the count.

**Interview check:** "How would you add a risk constraint?" Add CVaR variables `η` and
`z_s ≥ cost_s − η`, minimise `E[cost] + λ (η + mean(z_s) / (1 − α))`. Still an LP.

## 7. Heuristics

**Merit order** on the 2 × 2 toy by hand: cheapest unit first, up to its capacity.

In [34]:
order = units.sort_values("mc").index.tolist()
print("dispatch order:", order)
for h in [0, 1]:
    remaining = demand[h]
    line = f"hour {h}: demand {demand[h]:4.1f} ->"
    for g in order:
        take = min(units.loc[g, "p_max"], remaining)
        remaining = remaining - take
        line += f"  {g} = {take:4.1f}"
    print(line)

dispatch order: ['A', 'B']
hour 0: demand 12.0 ->  A = 10.0  B =  2.0
hour 1: demand  6.0 ->  A =  6.0  B =  0.0


Same schedule as the MILP found. Greedy ignores start-up costs and minimum up-times, so it
breaks when a unit would be switched on and off repeatedly. Check it against the exact
solution on a small instance before trusting it on a big one.

**`differential_evolution`** is a gradient-free global search for small, ugly objectives. A
1-D toy with two dips: the local optimiser from a bad start finds the wrong one, DE finds the global one.

In [35]:
def bumpy(x_):
    x_ = np.asarray(x_).ravel()[0]
    return (x_ - 2) ** 2 + 3 * np.sin(3 * x_)

from scipy.optimize import minimize
grid = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0])
print("f on a grid:", np.array([bumpy(g) for g in grid]).round(2))
local = minimize(bumpy, x0=[0.0], method="Nelder-Mead")
de = differential_evolution(bumpy, bounds=[(-2, 6)], seed=0)
print("Nelder-Mead from x0=0:", local.x.round(3), " f =", round(local.fun, 3))
print("differential_evolution:", de.x.round(3), " f =", round(de.fun, 3))

f on a grid: [ 4.    5.24  1.42 -2.68 -0.84  3.06  2.24 -0.39  2.39]
Nelder-Mead from x0=0: [-0.341]  f = 2.919
differential_evolution: [1.6]  f = -2.828


DE costs many function evaluations, so it is never the first choice for something `milp` or
`linprog` can solve. It is the right tool for a simulation-based objective with no structure.

## 8. Timing as the horizon grows

In [36]:
rows = []
for days in [1, 2, 4, 7]:
    dem = df["consumption_mwh"].values[:24 * days] / 1000
    c_, cons_, integ_, bnds_, names_ = build_uc(dem, units3)
    t0 = time.time()
    r_ = milp(c_, constraints=cons_, integrality=integ_, bounds=bnds_, options={"time_limit": 60})
    rows.append([days, len(names_), cons_.A.shape[0], r_.status, round(time.time() - t0, 2)])
pd.DataFrame(rows, columns=["days", "variables", "constraint rows", "status", "seconds"])

,days,variables,constraint rows,status,seconds
0,1,216,582,0,0.00
1,2,432,1446,0,0.01
2,4,864,3174,0,0.02
3,7,1512,5766,0,0.06


**Formulation tips**
- Real `p_max` in `p ≤ p_max · u`, never a huge M; variable bounds in `Bounds`, not as rows.
- Fewer binaries: `s` stays continuous, the constraints force it.
- Dense `A` is fine up to ~10⁴ rows; beyond that use `scipy.sparse.csr_matrix`.
- `options={"time_limit": ...}` and check `res.status == 1` for a time-limited answer; `res.mip_gap` says how far from proven optimal.
- No warm start in `scipy.optimize.milp`; rolling-horizon problems re-solve from scratch.

## Quick reference

| Task | Tool | Watch out |
|---|---|---|
| LP | `linprog(c, A_eq, b_eq, bounds, method="highs")` | `res.status`; duals in `res.eqlin.marginals` |
| MILP | `milp(c, constraints=LinearConstraint(A, lb, ub), integrality, bounds=Bounds(lo, hi))` | `integrality` is a float array; bounds separate from `A` |
| Variable positions | a name → column dict, printed as a table | most MILP bugs are indexing bugs |
| Judge the integer solution | solve the LP relaxation; gap = MILP / LP − 1 | rounding the LP is usually infeasible |
| Projects under a budget | knapsack: all-binary `milp` | greedy ratio is not always optimal |
| Decide under uncertainty | two-stage LP over scenarios | clip negative prices; evaluate out of sample; check `F*` stability |
| Risk | add CVaR variables, still an LP | choose `α`, `λ` deliberately |
| Too slow | merit order → local search → `differential_evolution` | verify against exact on a small instance |